In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_samples = 3000

# Generate raw features
data = {
    'Temperature': np.random.normal(25, 8, n_samples),
    'Humidity': np.random.uniform(10, 95, n_samples),
    'WindSpeed': np.random.exponential(5, n_samples),
    'HistoricalLoad': np.random.gamma(shape=2, scale=500, size=n_samples),
    'OperationalShift': np.random.choice(['Morning', 'Evening', 'Night', 'Maintenance'], n_samples, p=[0.4, 0.3, 0.2, 0.1]),
    'EquipmentAge': np.random.randint(1, 15, n_samples)
}

df = pd.DataFrame(data)

# Create a non-linear target condition with noise
load_risk = (df['HistoricalLoad'] * 0.001) + (df['Temperature'] * 0.3) - (df['Humidity'] * 0.05) + (df['OperationalShift'] == 'Evening') * 3
df['CriticalOverload'] = (load_risk > np.percentile(load_risk, 75)).astype(int) # Top 25% are overloads

# Inject real-world messiness (15% missing values in numeric fields) using a clean column-wise mask
cols_to_miss = ['Temperature', 'WindSpeed', 'HistoricalLoad']
for col in cols_to_miss:
    mask = np.random.rand(n_samples) < 0.15
    df.loc[mask, col] = np.nan

# Inject random typos in categorical strings
df.loc[np.random.choice(n_samples, 20, replace=False), 'OperationalShift'] = 'Mornng' # Typo

print(f"Dataset Shape: {df.shape}")
print(df.isnull().sum())

Dataset Shape: (3000, 7)
Temperature         417
Humidity              0
WindSpeed           466
HistoricalLoad      461
OperationalShift      0
EquipmentAge          0
CriticalOverload      0
dtype: int64


In [3]:
print(df.head)

<bound method NDFrame.head of       Temperature   Humidity  WindSpeed  HistoricalLoad OperationalShift  \
0             NaN  79.592920   1.238394             NaN            Night   
1       23.893886  93.603213   3.121374      412.251761            Night   
2             NaN  32.072021   2.839193     1398.778490      Maintenance   
3       37.184239  92.426509   8.039492     2093.405174          Evening   
4       23.126773  46.654879   6.725276     1473.184984      Maintenance   
...           ...        ...        ...             ...              ...   
2995    25.619844  36.591793   5.183058             NaN            Night   
2996    27.062020  18.333786        NaN      897.458560          Morning   
2997    15.065915  54.779459  13.321447      373.835887      Maintenance   
2998    27.673411  14.853422   7.682866      589.433326          Morning   
2999    23.757928  19.396169   1.115774     2695.963918          Evening   

      EquipmentAge  CriticalOverload  
0               11

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["CriticalOverload"]).values
y = df["CriticalOverload"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")

Shape of X_train: (2250, 6)
Shape of y_train: (2250,)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scalar', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('OneHot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, nu)
    ]
)

X_train_processsed = preprocessor(X_train)
y_train_processsed = preprocessor(y_train)

In [ ]:
import xgboost as xgb

dtrain = xgb.DMatrix(data = X_train_processsed, label = y_train)

params = {
    'objective' : 'binary:logistic',
    'eval_metric' : 'logloss',
    'learning_rate' : 0.01,
    'max_depth' : 6,
    'sub_sample' : .8 
}

cv_results = xgb.cv(
    params,
    dtrain,
    num_boost_round = 100,
    nfold = 5,
    early_stopping_round = 10,
    stratified = True,
    seed = 42
)

print(f"{cv_results.head}")

NameError: name 'X_train' is not defined